# Week 1, Assignment #1: Web Data Scraping

**Course:** Data & AI Specialist Program, Cyber Shujaa
**Opens:** Monday, 7 September 2026
**Due:** Sunday, 13 September 2026, 23:59
**Submit:** this notebook (.ipynb) plus your exported .csv file, via the file submission area on the LMS

## What this assignment is actually testing

In class we walked through the Data Science Methodology. This assignment is your first hands-on
pass through two of its steps: **Data Understanding** (going and getting real data instead of
being handed a clean file) and **Data Preparation** (fixing up what you collected before it is
usable). A live website is messier than any textbook dataset, which is exactly the point.

## Your target site (fixed, not a free choice)

Every student scrapes the same page for this assignment:
[scrapethissite.com/pages/simple/](https://www.scrapethissite.com/pages/simple/), "Countries of
the World: A Simple Example." It lists all 250 countries in the world, each with a capital,
population, and area. Using one fixed page keeps the assignment fair to grade and makes sure
everyone hits the same real-world snag: this page is **not** built as a `<table>`, unlike the
hockey site from the in-class demo, so you cannot reuse that code unchanged. You will apply the
same four-step pattern (fetch, parse, find the repeating structure, collect into a DataFrame) to
HTML shaped differently, which is the actual skill this assignment is testing.

## How this notebook works

This is a **scaffold, not a solution**. Some cells run as they are. Others have a `# TODO`
comment where you need to write real code yourself, using what you learned in the in-class demo
plus your own investigation of this page's HTML in DevTools. If a cell runs without errors but
you never replaced a `___` placeholder, you have not actually finished that step.

## Academic honesty

Everyone is scraping the same page, so it will be tempting to just copy a classmate's finished
selectors. Don't, the reflection questions at the end ask you to explain choices you made while
writing this code, and a script you did not actually write yourself will not let you answer them
honestly. Find the tags and classes below through your own inspection of the page.


## Step 1: Import your libraries

These three libraries do three different jobs in this pipeline: `requests` talks to the
website over HTTP, `BeautifulSoup` (from the `bs4` package) turns raw HTML text into something
you can search through, and `pandas` gives you the DataFrame you will store your results in.


In [ ]:
# Colab already has these installed, but this line is harmless if they are already present
# and saves you a confusing NameError if you are running this outside Colab.
!pip install requests beautifulsoup4 pandas --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Inspect your target page

Your target is fixed: `https://www.scrapethissite.com/pages/simple/`. Before writing any
scraping code, check whether scraping is allowed: visit
`https://www.scrapethissite.com/robots.txt` in your browser and read it.

**Do this now, before running the next cell:**
1. Open the target page in a normal browser tab.
2. Right-click on one of the 250 country blocks and choose "Inspect" (or press F12).
3. Find the HTML tag and class name that wraps **one single country** (not the whole page, not
   just the country's name). You will need this exact tag and class name in Step 5.
4. While you are in there, also find the tags used for the capital, population, and area inside
   one country block. You will need those in Step 6.


In [ ]:
url = "https://www.scrapethissite.com/pages/simple/"  # fixed target, same for everyone

# TODO: write one line below stating what you found at scrapethissite.com/robots.txt.
# Example: robots_txt_notes = "Disallow: /lessons/ and /faq/ only, /pages/ is not listed, safe to scrape."
robots_txt_notes = "Disallow: /lessons/ and /faq/ only, /pages/simple/ is not disallowed, so it is safe to scrape."

print(f"Target URL: {url}")
print(f"robots.txt check: {robots_txt_notes}")


Target URL: https://www.scrapethissite.com/pages/simple/
robots.txt check: Disallow: /lessons/ and /faq/ only, /pages/simple/ is not disallowed, so it is safe to scrape.


## Step 3: Fetch the page

`requests.get()` does not throw an error just because a page returns a 404 or a 500. It happily
hands you back a broken response and lets you find out the hard way three cells later when
`soup.find()` returns `None`. Catch that here instead.


In [ ]:
response = requests.get(url)

# TODO: requests.get() succeeded even if the server returned an error status. Add a check here:
# if response.status_code is not 200, print a clear message showing the actual status code and
# stop the notebook from continuing with a broken response.
# Hint: look up response.raise_for_status(), or write your own if-statement.
response.raise_for_status()

print(f"Status code: {response.status_code}")
print(f"Bytes received: {len(response.text)}")


Status code: 200
Bytes received: 203338


## Step 4: Parse the HTML

`response.text` is one enormous string. `BeautifulSoup` turns it into a tree you can actually
search through by tag and class, instead of doing string matching yourself.


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# Print the first 800 characters so you can see the structure you are about to search through.
# This should look like real, readable HTML, not an error page or a login wall.
print(soup.prettify()[:800])


<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping
  </title>
  <link href="/static/images/scraper-icon.png" rel="icon" type="image/png"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="A single page that lists information about all the countries in the world. Good for those just get started with web scraping." name="description"/>
  <link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" rel="stylesheet"/>



## Step 5: Find every repeating item on the page

This is the main challenge in this assignment. Every country on the page is wrapped in the same
repeating tag and class, the one you found with DevTools in Step 2. `soup.find_all()` returns
every element matching that tag and class as a list, all 250 of them.


In [ ]:
# TODO: replace both ___ with the real tag name and class name from your own inspection.
# Example shape only, not the real answer: soup.find_all("div", class_="___")
items = soup.find_all("div", class_="col-md-4 country")

print(f"Found {len(items)} items on the page.")

# TODO: the page header states how many countries are listed. Does your count match? If items
# is empty or the count looks wrong, your tag or class name is not matching what you think it
# is. Go back to DevTools and check again before moving on, extracting fields from an empty
# list will just waste the rest of this notebook.


Found 250 items on the page.


## Step 6: Extract the fields you actually care about

For each country found in Step 5, pull out four fields: **name, capital, population, and
area**. This is where most of the real work in this assignment lives: the exact inner tags and
classes for each field are yours to find in DevTools, the same way you found the repeating
container in Step 2.


In [ ]:
# show the first item so you can see the structure you are about to search through
items[0]

<div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag-icon-ad"></i>
                            Andorra
                        </h3>
<div class="country-info">
<strong>Capital:</strong> <span class="country-capital">Andorra la Vella</span><br/>
<strong>Population:</strong> <span class="country-population">84000</span><br/>
<strong>Area (km<sup>2</sup>):</strong> <span class="country-area">468.0</span><br/>
</div>
</div>

In [ ]:
scraped_rows = []

for item in items:
    # TODO: for each country block, find the specific inner tag holding each field you want and
    # pull out its text. Some patterns you will likely need:
    #   item.find("h3").text.strip()               #-> text inside a tag
    #   item.find("span", class_="country-capital").text.strip()  #-> text inside a tag with a specific class

    # Build a dictionary for this one country with clear column names, then add it to the list.
      row = {
            "name": item.find("h3", class_="country-name").text.strip(),
            "capital": item.find("span", class_="country-capital").text.strip(),
            "population": item.find("span", class_="country-population").text.strip(),
            "area_km2": item.find("span", class_="country-area").text.strip(),}
      scraped_rows.append(row)


# This check exists so you find out right here if Step 6 is still empty, instead of getting a
# confusing empty DataFrame three cells from now.
assert len(scraped_rows) > 0, "scraped_rows is still empty. Fill in the loop above before continuing."
print(f"Collected {len(scraped_rows)} rows.")


Collected 250 rows.


In [ ]:
scraped_rows[0]

{'name': 'Andorra',
 'capital': 'Andorra la Vella',
 'population': '84000',
 'area_km2': '468.0'}

## Step 7: Build your DataFrame

You built a tiny version of this in class from a single value. Here you are doing the same
thing at real scale, from a whole list of dictionaries instead of one.


In [ ]:
df = pd.DataFrame(scraped_rows)

print(df.shape)
df.head()


(250, 4)


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 8: Clean your data (Data Preparation)

Real scraped data is rarely usable as-is. This is step three of the Data Science Methodology
from class, and it is not optional here. At minimum, check for and handle each of the
following. Not every issue will exist in your data, but you need to actually check, not assume.


In [ ]:
# TODO 1: check for missing values. df.isnull().sum() will show you, column by column.
# Decide what to do about any you find (drop the row, fill a default value, or leave it with a
# one-line comment explaining why it is fine to leave).

print(df.isnull().sum())

name          0
capital       0
population    0
area_km2      0
dtype: int64


In [ ]:
# TODO 2: check for duplicate rows with df.duplicated().sum(). With 250 countries on one page
# this should be zero, if it isn't, your loop in Step 6 is probably reading something twice.
print(df.duplicated().sum())

0


In [ ]:
# TODO 3: check the data types with df.dtypes. population and area_km2 came out of the HTML as
# text, even though they are really numbers. Convert both to a real numeric type.
# Hint: pd.to_numeric(), or .astype(float) if the text is already clean digits.
print(df.dtypes)
df['population'] = pd.to_numeric(df['population'], errors='coerce')
df['area_km2'] = pd.to_numeric(df['area_km2'], errors='coerce')

name          object
capital       object
population    object
area_km2      object
dtype: object


In [ ]:
print(df.dtypes)
df.head()


name           object
capital        object
population      int64
area_km2      float64
dtype: object


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 9: Export to CSV

This is the deliverable file you submit alongside this notebook.


In [ ]:
# TODO: choose a clear filename, something like "assignment1_yourname.csv", not "data.csv".
output_filename = "assignment1_janenjuguna.csv"

df.to_csv(output_filename, index=False)

# Prove the export actually worked by reading the file back in, rather than just trusting that
# to_csv() did not raise an error.
check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()


Saved and reloaded 250 rows, 4 columns.


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 10: Reflection (required, answer in this cell)

Replace the placeholders below with your own two or three sentence answers.

1. **What was the hardest part of getting your selectors right in Step 5 or Step 6, and how did
   you eventually figure it out?**
   *Understanding the tags in the browser devtool was the hardest part but i eventually figured ow to get the tags and the classes*

2. **What did Step 8 reveal about your data that you did not expect before you actually looked
   at it closely?**
   *I didn't expect that the population and area_km2 were in string format before checking and learnt that scraped data is not automatically in the correct data type and should be checked and corrected*

3. **If you had another hour, what would you add or fix in this scraper?**
   *I would add more error handling and data validation to make the scraper more reliable*


## Submission checklist

- [ ] `robots_txt_notes` in Step 2 is filled in with your own real check, not a placeholder
- [ ] Step 3's status check actually stops on a failed request, not just prints a status code
- [ ] `items` in Step 5 is a non-empty list, roughly 250 items matching the page's own count
- [ ] `scraped_rows` in Step 6 has all four columns (name, capital, population, area) per row
- [ ] Step 8 shows evidence you actually checked for missing values, duplicates, and wrong types
- [ ] The exported .csv opens correctly when reloaded in Step 9
- [ ] All three reflection questions in Step 10 are answered in your own words
- [ ] You have submitted both this notebook (.ipynb) and your exported .csv file on the LMS

**Grading scale:** 0 = No submission, 1 = Below Expectation, 2 = Meets Expectation,
3 = Exceeds Expectation.